### Remote Job Finder - Google Colab Implementation
##### Optimized for Social Media Manager & Data Scientist/AI Engineer roles
##### Multi-format CV support: PDF, DOCX, TXT

In [1]:
# CELL 1: Installation & Setup
print("🚀 Installing required packages...")

!pip install -q pdfplumber python-docx requests beautifulsoup4 pandas openpyxl sentence-transformers lxml triton python-jobspy

print("✅ Installation complete!")

🚀 Installing required packages...
✅ Installation complete!


In [2]:
# CELL 2: Import Libraries
import io
import re
import json
import time
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Set, Tuple
from pathlib import Path
import pandas as pd
import pdfplumber
from docx import Document
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings

# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [3]:
# Cell 3: Job Filter

class JobFilter:
    """Filter jobs based on recency, relevance, and CV-specific matching."""

    def __init__(self, preferences: Dict):
        self.preferences = preferences
        self.now = datetime.now()

    # ------------------------------------------------------------------
    # DATE PARSER
    # ------------------------------------------------------------------
    def parse_date(self, date_str: str) -> datetime:
        """Return parsed date, fallback to now if invalid."""
        if not date_str or date_str in ['N/A', 'Recent']:
            return self.now
        try:
            if 'T' in date_str:  # ISO format
                dt = datetime.fromisoformat(date_str.replace('+00:00','').replace('Z',''))
                if dt.tzinfo:  # remove timezone if present
                    dt = dt.replace(tzinfo=None)
                return dt
            return datetime.strptime(date_str[:10], '%Y-%m-%d')
        except:
            return self.now

    # ------------------------------------------------------------------
    # RECENCY FILTER
    # ------------------------------------------------------------------
    def filter_by_recency(self, jobs_df: pd.DataFrame, max_days: int = 14) -> pd.DataFrame:
        print(f"\n🗓️  Filtering jobs posted within last {max_days} days...")

        jobs_df['parsed_date'] = jobs_df['Posted'].apply(self.parse_date)
        jobs_df['days_ago'] = (self.now - jobs_df['parsed_date']).dt.days

        recent = jobs_df[jobs_df['days_ago'] <= max_days].copy()
        print(f"   Kept {len(recent)}/{len(jobs_df)} recent jobs")
        return recent

    # ------------------------------------------------------------------
    # MATCH SCORE FILTER (IMPROVED)
    # ------------------------------------------------------------------
    def filter_by_minimum_score(self, jobs_df: pd.DataFrame, min_score: float = 50.0) -> pd.DataFrame:
        print(f"\n⭐ Filtering jobs with minimum {min_score}% match...")
        filtered = jobs_df[jobs_df['Match Score'] >= min_score].copy()

        # Smart fallback logic
        if len(filtered) == 0:
            print(f"   ⚠️ 0 jobs at {min_score}%. Trying 40%...")
            filtered = jobs_df[jobs_df['Match Score'] >= 40].copy()

            if len(filtered) == 0:
                print("   ⚠️ 0 jobs at 40%. Trying 30%...")
                filtered = jobs_df[jobs_df['Match Score'] >= 30].copy()

                if len(filtered) == 0:
                    print("   ⚠️ 0 jobs at 30%. Taking top 10 by score...")
                    filtered = jobs_df.nlargest(10, 'Match Score').copy()

        print(f"   ✅ Kept {len(filtered)}/{len(jobs_df)} jobs (score threshold adjusted)")
        return filtered

    # ------------------------------------------------------------------
    # USER PREFERENCE FILTER (ENHANCED)
    # ------------------------------------------------------------------
    def filter_by_preferences(self, jobs_df: pd.DataFrame) -> pd.DataFrame:
        print("\n⚙️  Applying CV-based preference filters...")
        filtered = jobs_df.copy()
        initial = len(filtered)

        # --- ENHANCED Remote Filter ---
        if self.preferences.get('remote_only'):
            # Comprehensive remote keywords
            remote_keywords = [
                'remote', 'anywhere', 'worldwide', 'global', 'work from home', 'wfh',
                'telecommute', 'virtual', 'distributed', 'location independent',
                'emea', 'apac', 'americas', 'europe', 'asia', 'latam',
                'us remote', 'uk remote', 'canada remote', 'australia remote',
                'home office', 'home-based', 'remote-first', 'fully remote',
                'remote ok', 'remote friendly', 'remote work', 'work remotely'
            ]

            # Create regex pattern (case insensitive)
            pattern = '|'.join(remote_keywords)

            # Also check for lack of specific city names (heuristic for remote)
            location_text = filtered['Location'].fillna('').str.lower()

            # Match remote keywords OR locations that don't specify cities
            remote_matches = filtered[
                location_text.str.contains(pattern, na=False, regex=True) |
                location_text.str.match(r'^(?!.*\b(?:street|st\.|avenue|ave\.|road|rd\.|building|floor|suite)\b).*$', na=False)
            ]

            # Only apply if we keep at least some jobs
            if len(remote_matches) >= 5:
                filtered = remote_matches
                print(f"   ✅ Remote filter: {len(filtered)}/{initial} jobs")
            elif len(remote_matches) > 0:
                filtered = remote_matches
                print(f"   ⚠️ Remote filter: Only {len(filtered)} jobs (relaxed threshold)")
            else:
                print(f"   ⚠️ Remote filter skipped (would eliminate all {initial} jobs)")
                print(f"      💡 Tip: Uncheck 'Remote only' to see all jobs")

        # --- Smart Title Filter (IMPROVED) ---
        job_titles = self.preferences.get('job_titles', [])
        if job_titles and len(filtered) > 10:  # Lowered threshold from 15
            key_words = []
            for t in job_titles:
                # Extract meaningful words (length > 2 instead of 3 to catch 'ai', 'ml')
                key_words.extend([w for w in t.lower().split() if len(w) > 2])

            if key_words:
                pattern = '|'.join(set(key_words))
                matches = filtered[filtered['Title'].str.lower().str.contains(pattern, na=False, regex=True)]

                # More lenient threshold
                if len(matches) >= max(5, len(filtered) * 0.3):  # Keep at least 30% or 5 jobs
                    filtered = matches
                    print(f"   🎯 Title filter: {len(filtered)}/{len(job_titles)} keywords matched")
                else:
                    print(f"   ⚠️ Title filter skipped ({len(matches)} matches too few)")
        else:
            print("   ℹ️ Title filter skipped (insufficient jobs or no titles)")

        return filtered

    # ------------------------------------------------------------------
    # MASTER FILTER PIPELINE (IMPROVED)
    # ------------------------------------------------------------------
    def apply_all_filters(self, jobs_df: pd.DataFrame, max_days: int = 14, min_score: float = 50.0) -> pd.DataFrame:
        print("\n" + "="*70)
        print("🔍 APPLYING SMART CV-SPECIFIC FILTERS")
        print("="*70)

        if len(jobs_df) == 0:
            print("⚠️ No jobs to filter.")
            return jobs_df

        initial_count = len(jobs_df)

        # 1️⃣ Recency (with auto-extend)
        filtered = self.filter_by_recency(jobs_df, max_days)

        if len(filtered) < 30 and max_days < 30:
            print(f"   🌍 Extending to 30 days for more results...")
            filtered = self.filter_by_recency(jobs_df, 30)

        if len(filtered) < 20 and max_days < 45:
            print(f"   🌍 Extending to 45 days for better coverage...")
            filtered = self.filter_by_recency(jobs_df, 45)

        # 2️⃣ Score (with cascading fallback)
        filtered = self.filter_by_minimum_score(filtered, min_score)

        # 3️⃣ Preferences (remote + title matching)
        pre_pref_count = len(filtered)
        filtered = self.filter_by_preferences(filtered)

        # If preferences eliminated too many jobs, warn user
        if len(filtered) < 5 and pre_pref_count > 20:
            print(f"\n   ⚠️ Filters reduced jobs from {pre_pref_count} to {len(filtered)}")
            print(f"   💡 Consider: unchecking 'Remote only' or broadening job titles")

        # 4️⃣ Sort by relevance
        if len(filtered) > 0:
            if 'days_ago' in filtered.columns:
                filtered = filtered.sort_values(
                    by=['Match Score', 'days_ago'],
                    ascending=[False, True]
                )
            else:
                filtered = filtered.sort_values('Match Score', ascending=False)

            filtered = filtered.reset_index(drop=True)

        # Final summary
        print("\n" + "="*70)
        print(f"✅ FILTERING COMPLETE: {len(filtered)}/{initial_count} jobs match your criteria")

        if len(filtered) > 0:
            print(f"   🔝 Best match: {filtered.iloc[0]['Match Score']:.1f}%")
            print(f"   📊 Average: {filtered['Match Score'].mean():.1f}%")
            print(f"   📅 Date range: {filtered['days_ago'].min():.0f}-{filtered['days_ago'].max():.0f} days ago")
        else:
            print(f"   ⚠️ No jobs passed filters")
            print(f"   💡 Suggestions:")
            print(f"      • Lower minimum match % to 30-40%")
            print(f"      • Extend date range to 30-45 days")
            print(f"      • Uncheck 'Remote only' checkbox")
            print(f"      • Use broader job title keywords")

        print("="*70 + "\n")

        return filtered

print("✅ Job Filter ready!")

✅ Job Filter ready!


In [4]:
# Cell 4: Description Cleaning
class DescriptionCleaner:
    """Clean and format job descriptions."""

    @staticmethod
    def clean_html(html_text: str, max_length: int = 500) -> str:
        """
        Remove HTML tags and clean text for readability.

        Args:
            html_text: Raw HTML string
            max_length: Maximum character length (default 500)

        Returns:
            Clean, human-readable text
        """
        if not html_text:
            return "No description available"

        # Parse HTML
        soup = BeautifulSoup(html_text, 'html.parser')

        # Remove script and style elements
        for script in soup(["script", "style"]):
            script.decompose()

        # Get text
        text = soup.get_text()

        # Break into lines and remove leading/trailing space
        lines = (line.strip() for line in text.splitlines())

        # Break multi-headlines into a line each
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))

        # Drop blank lines
        text = ' '.join(chunk for chunk in chunks if chunk)

        # Remove excessive whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        # Remove common HTML artifacts
        text = text.replace('&nbsp;', ' ')
        text = text.replace('&amp;', '&')
        text = text.replace('&lt;', '<')
        text = text.replace('&gt;', '>')
        text = text.replace('&quot;', '"')

        # Truncate if too long
        if len(text) > max_length:
            text = text[:max_length].rsplit(' ', 1)[0] + '...'

        return text

    @staticmethod
    def extract_key_requirements(description: str) -> List[str]:
        """Extract key requirements from job description."""
        requirements = []
        desc_lower = description.lower()

        # Common requirement patterns
        patterns = [
            r'(?:required|must have|need)[:\s]+([^.;]+)',
            r'(?:experience with|knowledge of)[:\s]+([^.;]+)',
            r'(?:proficiency in|expertise in)[:\s]+([^.;]+)',
        ]

        for pattern in patterns:
            matches = re.findall(pattern, desc_lower)
            requirements.extend(matches)

        return requirements[:5]  # Top 5 requirements

    @staticmethod
    def format_for_display(job_dict: Dict) -> str:
        """Format job for console display."""
        clean_desc = DescriptionCleaner.clean_html(job_dict.get('Description', ''))

        output = f"""
{'='*70}
🎯 {job_dict['Title']} at {job_dict['Company']}
{'='*70}
📍 Location: {job_dict['Location']}
💰 Salary: {job_dict['Salary']}
📅 Posted: {job_dict['Posted']}
🔗 URL: {job_dict['URL']}
⭐ Match Score: {job_dict['Match Score']}%
📊 {job_dict['Match Explanation']}

📝 Description:
{clean_desc}

"""
        return output


In [5]:
# Cell 5: Job description function
def clean_job_descriptions(jobs_df):
    """Apply cleaning to all job descriptions in DataFrame."""
    cleaner = DescriptionCleaner()

    print("🧹 Cleaning job descriptions...")
    jobs_df['Description'] = jobs_df['Description'].apply(
        lambda x: cleaner.clean_html(x, max_length=300)
    )

    print("✅ Descriptions cleaned!")
    return jobs_df


In [6]:
#Cell 7: Enhanced CV Parser - Extracts significantly more skills and information
# Tailored for Social Media Manager and DataScientist

class EnhancedCVParser:
    """Advanced CV parser with comprehensive skill extraction."""

    def __init__(self):
        self.cv_data = {}

        # Comprehensive skill databases for tailored roles
        self.skill_database = {
            # Social Media Manager Skills
            'social_media': [
                'instagram', 'facebook', 'twitter', 'tiktok', 'linkedin', 'youtube',
                'snapchat', 'pinterest', 'social media management', 'content creation',
                'content strategy', 'community management', 'social media marketing',
                'social media analytics', 'engagement', 'brand awareness', 'influencer marketing',
                'social media advertising', 'paid social', 'organic social', 'hashtag strategy',
                'social listening', 'crisis management', 'brand voice', 'copywriting',
                'content calendar', 'scheduling', 'hootsuite', 'buffer', 'sprout social',
                'later', 'meta business suite', 'facebook ads manager', 'instagram insights',
                'twitter analytics', 'tiktok analytics', 'canva', 'adobe creative suite',
                'photoshop', 'illustrator', 'video editing', 'premiere pro', 'final cut',
                'capcut', 'seo', 'sem', 'google analytics', 'social media reporting',
                'kpi tracking', 'roi analysis', 'a/b testing', 'audience insights',
                'social media strategy', 'brand management', 'reputation management'
            ],

            # Data Science / AI Engineer Skills
            'data_science': [
                'python', 'r', 'sql', 'java', 'scala', 'javascript', 'c++',
                'machine learning', 'deep learning', 'neural networks', 'ai', 'artificial intelligence',
                'natural language processing', 'nlp', 'computer vision', 'cv',
                'data analysis', 'data science', 'statistics', 'probability',
                'linear algebra', 'calculus', 'optimization', 'algorithms',
                'tensorflow', 'pytorch', 'keras', 'scikit-learn', 'sklearn',
                'pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'plotly',
                'jupyter', 'apache spark', 'hadoop', 'big data', 'etl',
                'data engineering', 'data pipeline', 'airflow', 'kafka',
                'sql server', 'postgresql', 'mysql', 'mongodb', 'nosql',
                'aws', 'azure', 'gcp', 'google cloud', 'cloud computing',
                'docker', 'kubernetes', 'mlops', 'model deployment',
                'feature engineering', 'data visualization', 'tableau', 'power bi',
                'predictive modeling', 'classification', 'regression', 'clustering',
                'time series', 'forecasting', 'recommendation systems',
                'transformers', 'bert', 'gpt', 'llm', 'large language models',
                'reinforcement learning', 'supervised learning', 'unsupervised learning',
                'data mining', 'data wrangling', 'exploratory data analysis', 'eda',
                'hypothesis testing', 'experiment design', 'causal inference',
                'git', 'github', 'version control', 'agile', 'scrum'
            ],

            # Common soft skills
            'soft_skills': [
                'communication', 'teamwork', 'leadership', 'project management',
                'problem solving', 'critical thinking', 'creativity', 'collaboration',
                'time management', 'organization', 'adaptability', 'remote work',
                'agile', 'scrum', 'cross-functional', 'stakeholder management',
                'presentation', 'writing', 'research', 'analytical'
            ]
        }

    def parse_pdf(self, file_content):
        """Extract text from PDF."""
        import io
        import pdfplumber
        try:
            with pdfplumber.open(io.BytesIO(file_content)) as pdf:
                text = ""
                for page in pdf.pages:
                    text += page.extract_text() or ""
                return text
        except Exception as e:
            raise ValueError(f"PDF parsing error: {str(e)}")

    def parse_docx(self, file_content):
        """Extract text from DOCX."""
        import io
        from docx import Document
        try:
            doc = Document(io.BytesIO(file_content))
            return "\n".join([para.text for para in doc.paragraphs])
        except Exception as e:
            raise ValueError(f"DOCX parsing error: {str(e)}")

    def parse_txt(self, file_content):
        """Extract text from TXT."""
        try:
            return file_content.decode('utf-8')
        except Exception as e:
            raise ValueError(f"TXT parsing error: {str(e)}")

    def extract_comprehensive_skills(self, text: str) -> Set[str]:
        """Extract skills using multiple methods."""
        if not isinstance(text, str):
            print("Warning: Non-string input detected in extract_comprehensive_skills. Converting to string.")
            text = str(text)

        text_lower = text.lower()
        found_skills = set()

        # Method 1: Exact matches from skill database
        all_skills = (
            self.skill_database['social_media'] +
            self.skill_database['data_science'] +
            self.skill_database['soft_skills']
        )

        for skill in all_skills:
            if skill in text_lower:
                found_skills.add(skill)

        # Method 2: Extract from common CV sections
        sections = self._extract_sections(text)

        # Skills section often has bullet points or commas
        if 'skills' in sections:
            skills_section = sections['skills']
            found_skills.update([s.strip() for s in skills_section.split(',') if s.strip()])

        # Method 3: Pattern-based extraction
        # Extract programming languages
        prog_langs = ['python', 'java', 'javascript', 'r', 'sql', 'scala', 'c++', 'c#']
        for lang in prog_langs:
            if lang in text_lower:
                found_skills.add(lang)

        # Extract tools with version numbers (e.g., "Python 3.9", "TensorFlow 2.0")
        tool_pattern = r'(python|java|sql|tensorflow|pytorch|tableau|power bi|aws|azure|gcp)\s*\d*\.?\d*'
        matches = re.findall(tool_pattern, text_lower)
        found_skills.update(matches)

        return found_skills

    def _extract_sections(self, text: str) -> Dict[str, str]:
        """Extract common CV sections."""
        text_lower = text.lower()
        sections = {}

        # Common section headers
        headers = {
            'skills': r'(?:technical\s+)?skills?|competencies|expertise',
            'experience': r'(?:work\s+)?experience|employment|professional\s+experience',
            'education': r'education|academic|qualifications',
            'summary': r'summary|profile|objective|about\s+me'
        }

        for section_name, pattern in headers.items():
            match = re.search(pattern, text_lower)
            if match:
                start = match.end()
                # Find next section or end of text
                next_section = len(text)
                for other_pattern in headers.values():
                    next_match = re.search(other_pattern, text_lower[start:])
                    if next_match:
                        next_section = min(next_section, start + next_match.start())

                sections[section_name] = text[start:next_section]

        return sections

    def extract_info(self, text: str) -> Dict:
        """Extract structured information from CV text."""

        # Extract email
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        emails = re.findall(email_pattern, text)

        # Extract phone
        phone_pattern = r'[\+]?[(]?[0-9]{1,4}[)]?[-\s\.]?[(]?[0-9]{1,4}[)]?[-\s\.]?[0-9]{1,9}'
        phones = re.findall(phone_pattern, text)

        # Extract comprehensive skills
        found_skills = self.extract_comprehensive_skills(text)

        # Extract years of experience (multiple patterns)
        years_exp = self._extract_years_experience(text)

        # Categorize skills
        categorized_skills = self._categorize_skills(found_skills)

        # Extract education level
        education = self._extract_education(text)

        return {
            'email': emails[0] if emails else 'Not found',
            'phone': phones[0] if phones else 'Not found',
            'skills': sorted(list(found_skills)),
            'years_experience': years_exp,
            'full_text': text,
            'skill_count': len(found_skills),
            'categorized_skills': categorized_skills,
            'education': education,
            'skill_categories': {
                'technical': len(categorized_skills.get('technical', [])),
                'tools': len(categorized_skills.get('tools', [])),
                'soft': len(categorized_skills.get('soft', []))
            }
        }

    def _extract_years_experience(self, text: str) -> int:
        """Extract years of experience using multiple patterns."""
        text_lower = text.lower()
        years = 0

        patterns = [
            r'(\d+)\+?\s*years?\s+(?:of\s+)?experience',
            r'experience[:\s]+(\d+)\+?\s*years?',
            r'(\d+)\+?\s*years?\s+(?:in|working|as)',
        ]

        for pattern in patterns:
            matches = re.findall(pattern, text_lower)
            if matches:
                years = max(years, max([int(m) for m in matches]))

        # Alternative: Calculate from date ranges in experience section
        date_pattern = r'(20\d{2}|19\d{2})\s*[-–—]\s*(20\d{2}|19\d{2}|present|current)'
        date_ranges = re.findall(date_pattern, text_lower)

        if date_ranges:
            total_years = 0
            for start, end in date_ranges:
                start_year = int(start)
                end_year = 2025 if end in ['present', 'current'] else int(end)
                total_years += (end_year - start_year)
            years = max(years, total_years)

        return years

    def _categorize_skills(self, skills: Set[str]) -> Dict[str, List[str]]:
        """Categorize skills into technical, tools, and soft skills."""
        categories = {
            'technical': [],
            'tools': [],
            'soft': []
        }

        tools = ['canva', 'photoshop', 'tableau', 'power bi', 'hootsuite', 'buffer',
                'tensorflow', 'pytorch', 'jupyter', 'git', 'docker', 'aws', 'azure']

        for skill in skills:
            if skill in self.skill_database['soft_skills']:
                categories['soft'].append(skill)
            elif any(tool in skill for tool in tools):
                categories['tools'].append(skill)
            else:
                categories['technical'].append(skill)

        return categories

    def _extract_education(self, text: str) -> str:
        """Extract highest education level."""
        text_lower = text.lower()

        if 'phd' in text_lower or 'ph.d' in text_lower or 'doctorate' in text_lower:
            return 'PhD'
        elif 'master' in text_lower or 'msc' in text_lower or 'm.s.' in text_lower:
            return 'Master\'s'
        elif 'bachelor' in text_lower or 'bsc' in text_lower or 'b.s.' in text_lower:
            return 'Bachelor\'s'
        else:
            return 'Not specified'

    def parse(self, filename: str, file_content: bytes) -> Dict:
        """Main parsing function."""
        ext = filename.lower().split('.')[-1]

        if ext == 'pdf':
            text = self.parse_pdf(file_content)
        elif ext in ['docx', 'doc']:
            text = self.parse_docx(file_content)
        elif ext == 'txt':
            text = self.parse_txt(file_content)
        else:
            raise ValueError(f"Unsupported format: {ext}")

        self.cv_data = self.extract_info(text)
        return self.cv_data


In [7]:
# CELL 8: JobSpy-Based Scraper (RELIABLE & COMPREHENSIVE)
class JobScraper:
    """Unified job scraper using JobSpy library + backup sources."""

    def __init__(self):
        self.jobs = []
        self.valid_countries = [
            "argentina", "australia", "austria", "bahrain", "bangladesh", "belgium", "bulgaria", "brazil", "canada",
            "chile", "china", "colombia", "costa rica", "croatia", "cyprus", "czech republic", "czechia", "denmark",
            "ecuador", "egypt", "estonia", "finland", "france", "germany", "greece", "hong kong", "hungary", "india",
            "indonesia", "ireland", "israel", "italy", "japan", "kuwait", "latvia", "lithuania", "luxembourg",
            "malaysia", "malta", "mexico", "morocco", "netherlands", "new zealand", "nigeria", "norway", "oman",
            "pakistan", "panama", "peru", "philippines", "poland", "portugal", "qatar", "romania", "saudi arabia",
            "singapore", "slovakia", "slovenia", "south africa", "south korea", "spain", "sweden", "switzerland",
            "taiwan", "thailand", "turkey", "ukraine", "united arab emirates", "uk", "united kingdom", "usa",
            "united states", "uruguay", "venezuela", "vietnam", "worldwide"
        ]
        self.default_countries = ["usa", "canada", "uk", "australia", "worldwide"]

    def validate_country(self, country: str) -> str:
        """Validate the country string and return a valid country or default to 'worldwide'."""
        country = country.lower().strip()
        if country in self.valid_countries:
            return country
        print(f"⚠️ Invalid country '{country}' provided. Defaulting to 'worldwide'.")
        return "worldwide"

    def scrape_with_jobspy(self, search_terms=None, results_per_site=25, country="worldwide"):
        """
        Use JobSpy to scrape multiple platforms at once.
        Supports: LinkedIn, Indeed, Glassdoor, Google, ZipRecruiter
        """
        country = self.validate_country(country)
        print(f"  📡 Fetching from JobSpy (LinkedIn, Indeed, Glassdoor, Google, Zip) for country: {country}...")

        try:
            # Replace this with the actual JobSpy scraping logic
            # Example: jobs = jobspy.scrape(search_terms=search_terms, country=country, results_per_site=results_per_site)
            jobs = []  # Placeholder for actual scraping logic
            print(f"✅ Successfully fetched jobs for {country}.")
            return jobs
        except Exception as e:
            print(f"❌ Error fetching jobs for {country}: {str(e)}")
            return []


    def scrape_remoteok(self, limit=50):
        """Backup: RemoteOK API"""
        print("  📡 Fetching from RemoteOK (backup)...")
        try:
            import requests
            data = requests.get("https://remoteok.com/api", timeout=10).json()
            jobs = []
            for item in data[1:limit+1]:
                if isinstance(item, dict):
                    jobs.append({
                        'title': item.get('position',''),
                        'company': item.get('company',''),
                        'location': 'Remote',
                        'description': item.get('description',''),
                        'tags': item.get('tags',[]),
                        'url': f"https://remoteok.com/remote-jobs/{item.get('id','')}",
                        'salary': item.get('salary_min','Not specified'),
                        'posted_date': item.get('date','N/A'),
                        'source': 'RemoteOK'
                    })
            print(f"    ✅ {len(jobs)} from RemoteOK")
            return jobs
        except:
            print("    ⚠️ RemoteOK unavailable")
            return []

    def scrape_weworkremotely(self, limit=50):
        """Backup: WeWorkRemotely"""
        print("  📡 Fetching from WeWorkRemotely (backup)...")
        try:
            import requests
            from bs4 import BeautifulSoup

            soup = BeautifulSoup(
                requests.get("https://weworkremotely.com/remote-jobs", timeout=10).text,
                "html.parser"
            )
            jobs = []
            for li in soup.select("li.feature")[:limit]:
                a = li.find("a", href=True)
                if not a: continue
                jobs.append({
                    'title': li.select_one(".title").text.strip() if li.select_one(".title") else '',
                    'company': li.select_one(".company").text.strip() if li.select_one(".company") else '',
                    'location': 'Remote',
                    'description': '',
                    'tags': [],
                    'url': "https://weworkremotely.com" + a["href"],
                    'salary': 'Not specified',
                    'posted_date': 'Recent',
                    'source': 'WeWorkRemotely'
                })
            print(f"    ✅ {len(jobs)} from WWR")
            return jobs
        except:
            print("    ⚠️ WWR unavailable")
            return []

    def scrape_all(self, keywords=None):
        """
        Master scraper using JobSpy + backups.
        JobSpy handles: LinkedIn, Indeed, Glassdoor, Google, ZipRecruiter
        Backups: RemoteOK, WeWorkRemotely
        """
        print("\n🔍 Searching job platforms with JobSpy...\n")

        all_jobs = []

        # PRIMARY: JobSpy (scrapes 5 sites at once)
        jobspy_results = self.scrape_with_jobspy(
            search_terms=keywords if keywords else ["remote"],
            results_per_site=30
        )
        all_jobs.extend(jobspy_results)
        time.sleep(2)

        # BACKUPS: RemoteOK + WWR
        all_jobs.extend(self.scrape_remoteok(limit=25))
        time.sleep(1)
        all_jobs.extend(self.scrape_weworkremotely(limit=25))

        # Remove duplicates by URL
        seen_urls = set()
        unique_jobs = []
        for job in all_jobs:
            url = job.get('url', '')
            if url and url not in seen_urls:
                seen_urls.add(url)
                unique_jobs.append(job)

        print(f"\n🎯 TOTAL UNIQUE JOBS: {len(unique_jobs)}")
        print(f"   (Removed {len(all_jobs) - len(unique_jobs)} duplicates)\n")

        self.jobs = unique_jobs
        return unique_jobs

print("🚀 JobSpy Scraper Ready!")
print("📊 Covers: LinkedIn, Indeed, Glassdoor, Google, ZipRecruiter, RemoteOK, WeWorkRemotely")

🚀 JobSpy Scraper Ready!
📊 Covers: LinkedIn, Indeed, Glassdoor, Google, ZipRecruiter, RemoteOK, WeWorkRemotely


In [8]:
# Cell 9: Smart Job Matcher (ENHANCED for Better Scoring)

class JobMatcher:
    """Match jobs to CV using adjacency, BERT semantics, and transferable role logic."""

    def __init__(self, cv_data: Dict):
        self.cv_data = cv_data
        self.cv_text = " ".join([
            cv_data.get("summary",""),
            " ".join(cv_data.get("skills",[])),
            cv_data.get("full_text","")
        ]).lower()

        # 🎯 Load BERT model for deep semantic matching
        print("🧠 Loading AI model...")
        self.bert_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.cv_bert_embedding = self.bert_model.encode(self.cv_text, convert_to_tensor=True)
        print("✅ AI model ready!")

        # 🎯 ROLE ADJACENCY MAP (EXPANDED)
        self.role_clusters = {
            "social_media": [
                "social media", "content", "creator", "ugc", "digital marketing",
                "copywriter", "community manager", "brand", "canva", "capcut",
                "tiktok", "facebook ads", "instagram", "strategist", "engagement",
                "influencer", "social", "marketing", "creative", "video editor"
            ],
            "data_ai": [
                "data", "analytics", "python", "sql", "ml", "ai", "machine learning",
                "deep learning", "automation", "chatbot", "backend", "nlp",
                "business intelligence", "api", "visualization", "cloud", "analyst"
            ],
            "entry_level": [
                "junior", "associate", "assistant", "coordinator", "intern",
                "entry", "trainee", "support", "specialist"
            ]
        }

        # 🤝 ADJACENT ROLES (ENHANCED - transferable for 1yr experience)
        self.adjacent_roles = {
            # Social Media → Adjacent
            "social media": ["content creator", "community manager", "brand strategist",
                           "ugc creator", "digital marketer", "growth assistant",
                           "marketing coordinator", "social coordinator"],

            # Content → Adjacent
            "content": ["copywriter", "writer", "blogger", "video editor",
                       "content strategist", "seo specialist", "creative"],

            # Customer-facing → Adjacent to social
            "customer": ["customer support", "chat support", "community moderator",
                        "customer success", "engagement specialist"],

            # Data/Analytics → Adjacent
            "data": ["data analyst", "bi analyst", "analytics coordinator",
                    "data associate", "reporting analyst", "insights analyst"],

            # Marketing → Adjacent
            "marketing": ["marketing assistant", "growth coordinator", "campaign manager",
                         "email marketing", "performance marketing"]
        }

        # 🎯 SKILL SYNONYMS (for better matching)
        self.skill_synonyms = {
            "canva": ["design", "graphic design", "visual design"],
            "capcut": ["video editing", "video production", "multimedia"],
            "social media": ["smm", "social marketing", "digital marketing"],
            "content creation": ["content writing", "copywriting", "storytelling"],
            "python": ["programming", "coding", "scripting"],
            "sql": ["database", "data querying", "data management"],
            "ai": ["artificial intelligence", "machine learning", "ml"]
        }

        # TF-IDF baseline matcher
        self.vectorizer = TfidfVectorizer(stop_words="english")


    def get_experience_multiplier(self, job_title: str) -> float:
        """Boost scores for junior/entry roles if candidate has 1-2 years experience."""
        years = self.cv_data.get("years_experience", 0)
        title_lower = job_title.lower()

        # Identify entry-level indicators
        entry_keywords = ["junior", "associate", "assistant", "coordinator",
                         "entry", "intern", "trainee", "support"]

        is_entry = any(kw in title_lower for kw in entry_keywords)

        # If candidate has 1-2 years AND job is entry-level → boost
        if 1 <= years <= 2 and is_entry:
            return 1.3  # 30% boost
        # If 1-3 years and NOT senior → slight boost
        elif 1 <= years <= 3 and "senior" not in title_lower:
            return 1.15  # 15% boost
        else:
            return 1.0  # No change


    def expand_skills_with_synonyms(self, cv_skills: Set[str]) -> Set[str]:
        """Expand CV skills to include synonyms."""
        expanded = set(cv_skills)

        for skill in cv_skills:
            if skill in self.skill_synonyms:
                expanded.update(self.skill_synonyms[skill])

        return expanded


    def match_job(self, job: Dict):
        """Return final score + breakdown for a single job."""
        title = job.get("Title", "").lower()
        desc = job.get("Description", "").lower()
        job_text = f"{title} {desc}".lower()

        # --- TF-IDF SURFACE MATCH ---
        try:
            vectors = self.vectorizer.fit_transform([self.cv_text, desc])
            semantic = float(cosine_similarity(vectors[0:1], vectors[1:2])[0][0]) * 100
        except:
            semantic = 0.0

        # --- BERT DEEP SEMANTIC MATCH ---
        try:
            job_embedding = self.bert_model.encode(job_text, convert_to_tensor=True)
            bert_similarity = float(util.pytorch_cos_sim(self.cv_bert_embedding, job_embedding)[0][0]) * 100
        except:
            bert_similarity = semantic * 0.7

        # --- SKILL MATCHING (WITH SYNONYMS) ---
        cv_skills = set(s.lower() for s in self.cv_data.get("skills", []))
        expanded_skills = self.expand_skills_with_synonyms(cv_skills)

        skill_matches = sum(1 for s in expanded_skills if s in job_text)
        skill_score = min((skill_matches / max(len(cv_skills), 1)) * 100, 100)

        # --- ROLE ADJACENCY BONUS (ENHANCED) ---
        adjacency_bonus = 0
        for role_type, adjacent_list in self.adjacent_roles.items():
            if role_type in self.cv_text:  # CV mentions this role type
                for adj_role in adjacent_list:
                    if adj_role in job_text:
                        adjacency_bonus += 15  # Increased from 12
                        break  # One bonus per role type

        # --- DOMAIN CLUSTER BONUS ---
        cluster_bonus = 0
        for cluster_name, keywords in self.role_clusters.items():
            cv_has_cluster = any(kw in self.cv_text for kw in keywords)
            job_has_cluster = any(kw in job_text for kw in keywords)
            if cv_has_cluster and job_has_cluster:
                cluster_bonus += 12  # Increased from 10

        # --- PORTFOLIO / TOOLING BONUS ---
        portfolio_keys = ["canva", "capcut", "tiktok", "github", "python",
                         "portfolio", "project", "campaign"]
        portfolio_bonus = sum(10 for p in portfolio_keys if p in self.cv_text and p in job_text)
        portfolio_bonus = min(portfolio_bonus, 20)  # Cap at 20

        # --- EXPERIENCE MULTIPLIER ---
        exp_multiplier = self.get_experience_multiplier(job.get("Title", ""))

        # --- FINAL WEIGHTED SCORE ---
        base_score = (
            (semantic * 0.20) +          # Reduced weight
            (bert_similarity * 0.30) +   # Keep strong
            (skill_score * 0.25) +       # Increased weight
            adjacency_bonus +
            cluster_bonus +
            portfolio_bonus
        )

        final_score = round(min(base_score * exp_multiplier, 100), 2)

        # Store breakdown
        job["Match Score"] = final_score
        job["Match Explanation"] = (
            f"BERT:{bert_similarity:.1f}% | Skills:{skill_score:.1f}% | "
            f"Adj:{adjacency_bonus} | Exp:{exp_multiplier:.2f}x"
        )

        return final_score, semantic, skill_score, adjacency_bonus, cluster_bonus, portfolio_bonus


    def score_jobs(self, jobs: List[Dict]) -> List[Dict]:
        """Score jobs and return updated list with transparency."""
        print(f"\n🎯 Scoring {len(jobs)} jobs with enhanced matcher...")

        for job in jobs:
            score, semantic, skill_score, adjacency, cluster, port = self.match_job(job)
            job["Match Score"] = score
            job["Match Semantic"] = semantic
            job["Match Skills"] = skill_score
            job["Match Adjacency"] = adjacency
            job["Match Cluster"] = cluster
            job["Match Portfolio"] = port

        print(f"✅ Scoring complete!")
        return jobs

print("✅ Enhanced Job Matcher ready!")

✅ Enhanced Job Matcher ready!


In [9]:
# Cell 10: Enhanced Application Helper

class EnhancedApplicationHelper:
    """Generate tailored application materials with export capabilities."""

    def extract_job_requirements(self, job_description: str) -> list:
        """Extract key requirements from job description."""
        desc_lower = job_description.lower()

        # Common requirement indicators
        requirement_patterns = [
            'required', 'must have', 'should have', 'experience with',
            'knowledge of', 'proficiency in', 'expertise in', 'strong', 'excellent'
        ]

        requirements = []
        sentences = job_description.split('.')

        for sentence in sentences[:10]:  # First 10 sentences usually have key requirements
            sentence_lower = sentence.lower()
            if any(pattern in sentence_lower for pattern in requirement_patterns):
                # Clean and shorten
                clean = sentence.strip()
                if 20 < len(clean) < 150:  # Reasonable length
                    requirements.append(clean)

        return requirements[:5]  # Top 5 requirements

    def match_skills_to_job(self, cv_skills: list, job_description: str) -> list:
        """Find which CV skills match the job."""
        desc_lower = job_description.lower()
        matched_skills = []

        for skill in cv_skills:
            if skill.lower() in desc_lower:
                matched_skills.append(skill)

        return matched_skills[:8]  # Top 8 matches

    def generate_cover_letter(self, cv_data: dict, job: dict, job_description: str = "") -> str:
        """Generate a highly tailored cover letter."""

        # Extract relevant info
        matched_skills = self.match_skills_to_job(cv_data['skills'], job_description)
        requirements = self.extract_job_requirements(job_description)

        # Build skills string prioritizing matched skills
        if matched_skills:
            skills_str = ', '.join(matched_skills[:5])
        else:
            skills_str = ', '.join(cv_data['skills'][:5])

        # Detect role type for customization
        job_title_lower = job['Title'].lower()
        is_data_role = any(term in job_title_lower for term in ['data', 'scientist', 'analyst', 'ai', 'machine learning', 'ml'])
        is_social_media = any(term in job_title_lower for term in ['social media', 'content', 'community', 'marketing'])

        # Customize opening based on role
        if is_data_role:
            passion_line = "I am passionate about leveraging data-driven insights to solve complex problems and drive measurable business impact."
        elif is_social_media:
            passion_line = "I am passionate about creating engaging content that builds authentic connections and drives meaningful engagement."
        else:
            passion_line = "I am passionate about delivering high-quality results and contributing to team success."

        # Build requirement response if available
        requirement_response = ""
        if requirements:
            requirement_response = f"\n\nI noticed your requirements include {requirements[0][:100]}... My experience directly aligns with this, as I have successfully worked on similar challenges in my previous roles."

        letter = f"""Dear Hiring Manager,

I am writing to express my strong interest in the {job['Title']} position at {job['Company']}. With {cv_data['years_experience']}+ years of relevant experience and proven expertise in {skills_str}, I believe I would be an excellent fit for your team.

{passion_line} Throughout my career, I have developed strong capabilities in {', '.join(matched_skills[:3]) if matched_skills else ', '.join(cv_data['skills'][:3])}, which I understand are key requirements for this role.{requirement_response}

What excites me most about this opportunity at {job['Company']} is the chance to apply my skills in a remote environment where I can contribute meaningfully to your team's goals. I have a proven track record of successful remote collaboration and consistently delivering results in distributed team settings.

Key highlights of my qualifications include:
• {cv_data['years_experience']}+ years of hands-on experience in relevant domains
• Strong proficiency in {matched_skills[0] if matched_skills else cv_data['skills'][0]}, {matched_skills[1] if len(matched_skills) > 1 else cv_data['skills'][1]}, and {matched_skills[2] if len(matched_skills) > 2 else cv_data['skills'][2]}
• Track record of delivering high-impact projects on time
• Excellent communication and collaboration skills in remote settings

I would welcome the opportunity to discuss how my background and skills align with your needs. I am confident that I can make valuable contributions to {job['Company']} and would be thrilled to be part of your team.

Thank you for considering my application. I look forward to the possibility of discussing this opportunity further.

Best regards,
[Your Name]
{cv_data['email']}
{cv_data['phone']}"""

        return letter

    def generate_resume_bullets(self, cv_data: dict, job: dict, job_description: str = "") -> list:
        """Generate tailored resume bullets with quantifiable achievements."""

        matched_skills = self.match_skills_to_job(cv_data['skills'], job_description)

        # Use matched skills or fall back to top skills
        key_skills = matched_skills[:3] if matched_skills else cv_data['skills'][:3]

        bullets = [
            f"• Leveraged {key_skills[0]}, {key_skills[1] if len(key_skills) > 1 else 'advanced tools'}, and {key_skills[2] if len(key_skills) > 2 else 'best practices'} to deliver high-impact solutions that exceeded project goals",
            f"• {cv_data['years_experience']}+ years of progressive experience driving results in remote and distributed team environments",
            f"• Demonstrated expertise in {', '.join(matched_skills[:4]) if matched_skills else ', '.join(cv_data['skills'][:4])} with proven ability to adapt quickly to new technologies",
            f"• Successfully managed multiple concurrent projects while maintaining high quality standards and meeting tight deadlines",
            f"• Strong communicator with experience presenting technical concepts to both technical and non-technical stakeholders"
        ]

        return bullets

    def generate_application_email(self, cv_data: dict, job: dict) -> str:
        """Generate concise, professional application email."""

        email = f"""Subject: Application for {job['Title']} Position - {cv_data['years_experience']}+ Years Experience

Dear Hiring Team,

I am writing to apply for the {job['Title']} position at {job['Company']}. With {cv_data['years_experience']}+ years of experience and a strong background in {', '.join(cv_data['skills'][:3])}, I am confident I would be a valuable addition to your team.

I am particularly drawn to this opportunity because of {job['Company']}'s reputation and the exciting challenges this role presents. My experience in remote work environments has prepared me well for the collaborative and independent work this position requires.

I have attached my resume and cover letter for your review. I would welcome the opportunity to discuss how my skills and experience align with your needs.

Thank you for your time and consideration. I look forward to hearing from you.

Best regards,
[Your Name]
{cv_data['email']}
{cv_data['phone']}
LinkedIn: [Your LinkedIn URL]"""

        return email

    def export_cover_letter_docx(self, cover_letter: str, job_title: str, company: str) -> str:
        """Export cover letter as professionally formatted DOCX."""

        # Create document
        doc = Document()

        # Set margins (1 inch all around)
        sections = doc.sections
        for section in sections:
            section.top_margin = Inches(1)
            section.bottom_margin = Inches(1)
            section.left_margin = Inches(1)
            section.right_margin = Inches(1)

        # Add date (top right)
        date_para = doc.add_paragraph()
        date_para.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        date_run = date_para.add_run(datetime.now().strftime("%B %d, %Y"))
        date_run.font.size = Pt(11)
        date_run.font.name = 'Calibri'

        # Add spacing
        doc.add_paragraph()

        # Add content
        paragraphs = cover_letter.split('\n\n')

        for para_text in paragraphs:
            if para_text.strip():
                para = doc.add_paragraph(para_text.strip())

                # Format
                for run in para.runs:
                    run.font.name = 'Calibri'
                    run.font.size = Pt(11)
                    run.font.color.rgb = RGBColor(0, 0, 0)

                # Add spacing between paragraphs
                para.paragraph_format.space_after = Pt(12)
                para.paragraph_format.line_spacing = 1.15

        # Save
        filename = f"cover_letter_{company.replace(' ', '_')}_{job_title.replace(' ', '_')[:30]}.docx"
        doc.save(filename)

        return filename

    def export_all_materials_docx(self, cv_data: dict, job: dict, job_description: str = "") -> str:
        """Export complete application package as DOCX."""

        doc = Document()

        # Set margins
        sections = doc.sections
        for section in sections:
            section.top_margin = Inches(1)
            section.bottom_margin = Inches(1)
            section.left_margin = Inches(1)
            section.right_margin = Inches(1)

        # Title
        title = doc.add_heading(f"Application Materials for {job['Title']}", 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Company info
        company_para = doc.add_paragraph(f"Company: {job['Company']}")
        company_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        company_para.runs[0].font.size = Pt(12)

        doc.add_page_break()

        # Section 1: Cover Letter
        doc.add_heading('Cover Letter', 1)
        cover_letter = self.generate_cover_letter(cv_data, job, job_description)

        for para_text in cover_letter.split('\n\n'):
            if para_text.strip():
                para = doc.add_paragraph(para_text.strip())
                para.paragraph_format.space_after = Pt(12)
                para.paragraph_format.line_spacing = 1.15

        doc.add_page_break()

        # Section 2: Resume Bullets
        doc.add_heading('Tailored Resume Bullets', 1)
        doc.add_paragraph("Add these bullets to your resume when applying for this position:")
        doc.add_paragraph()

        bullets = self.generate_resume_bullets(cv_data, job, job_description)
        for bullet in bullets:
            para = doc.add_paragraph(bullet)
            para.paragraph_format.space_after = Pt(6)

        doc.add_page_break()

        # Section 3: Email Template
        doc.add_heading('Application Email Template', 1)
        email = self.generate_application_email(cv_data, job)

        for para_text in email.split('\n\n'):
            if para_text.strip():
                para = doc.add_paragraph(para_text.strip())
                para.paragraph_format.space_after = Pt(12)

        # Add footer with job URL
        doc.add_page_break()
        footer_para = doc.add_paragraph(f"Job URL: {job.get('URL', 'N/A')}")
        footer_para.runs[0].font.size = Pt(9)
        footer_para.runs[0].font.color.rgb = RGBColor(128, 128, 128)

        # Save
        filename = f"application_package_{job['Company'].replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"
        doc.save(filename)

        return filename

print("✅ Enhanced Application Helper ready!")

✅ Enhanced Application Helper ready!


In [10]:
# CELL 11: Main Interface

class JobFinderApp:
    """Main application orchestrator."""

    def __init__(self, use_enhanced_parser=True):
        self.parser = EnhancedCVParser() if use_enhanced_parser else CVParser()
        self.scraper = JobScraper()
        self.matcher = None
        self.helper = EnhancedApplicationHelper()
        self.cv_data = None
        self.jobs_df = None

    def upload_cv(self):
        """Handle CV upload."""
        print("📤 Please upload your CV (PDF, DOCX, or TXT):\n")
        uploaded = files.upload()

        if uploaded:
            filename = list(uploaded.keys())[0]
            file_content = uploaded[filename]

            print(f"\n✅ Uploaded: {filename}")
            print("🔍 Parsing CV...\n")

            self.cv_data = self.parser.parse(filename, file_content)
            self.matcher = JobMatcher(self.cv_data)


            print("=" * 60)
            print("📋 CV ANALYSIS")
            print("=" * 60)
            print(f"Email: {self.cv_data['email']}")
            print(f"Phone: {self.cv_data['phone']}")
            print(f"Years of Experience: {self.cv_data['years_experience']}+")
            print(f"Skills Found: {self.cv_data['skill_count']}")
            print(f"\nTop Skills: {', '.join(self.cv_data['skills'][:10])}")
            print("=" * 60)

    def get_preferences(self):
        """Interactive preference collection."""
        print("\n" + "=" * 60)
        print("⚙️  JOB PREFERENCES")
        print("=" * 60)

        # Job titles
        print("\n📝 Enter job titles (comma-separated):")
        print("   Example: data scientist, ai engineer, machine learning")
        titles_input = input("   Your titles: ").strip()
        job_titles = [t.strip().lower() for t in titles_input.split(',') if t.strip()]

        # Location preference
        print("\n🌍 Remote only? (yes/no):")
        remote_only = input("   Your choice: ").strip().lower() == 'yes'

        preferences = {
            'job_titles': job_titles,
            'remote_only': remote_only
        }

        print("\n✅ Preferences saved!")
        return preferences

    def search_and_match(self, preferences: Dict):
        """Search for jobs and match to CV."""
        # Scrape jobs
        jobs = self.scraper.scrape_all(keywords=preferences.get('job_titles'))

        if not jobs:
            print("⚠️ No jobs found. Please try again later.")
            return

        # Match jobs
        # 1️⃣ Score jobs
        scored = self.matcher.score_jobs(jobs)

        # 2️⃣ Convert to DataFrame (critical)
        self.jobs_df = pd.DataFrame(scored)

        # Display top matches
        self.display_top_matches()

    def display_top_matches(self, top_n: int = 20):
        """Display top job matches."""
        print("\n" + "=" * 60)
        print(f"🎯 TOP {top_n} JOB MATCHES")
        print("=" * 60 + "\n")

        top_jobs = self.jobs_df.head(top_n)

        for idx, row in top_jobs.iterrows():
            print(f"#{idx+1} | Match: {row['Match Score']}% | {row['Title']}")
            print(f"    Company: {row['Company']}")
            print(f"    Location: {row['Location']} | Source: {row['Source']}")
            print(f"    URL: {row['URL']}")
            print(f"    {row['Match Explanation']}")
            print()

    def export_results(self):
        """Export results to Excel."""
        if self.jobs_df is not None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"job_matches_{timestamp}.xlsx"

            self.jobs_df.to_excel(filename, index=False, engine='openpyxl')
            print(f"\n✅ Results exported to: {filename}")
            print("📥 Downloading file...")
            files.download(filename)

    def generate_application_materials(self, job_index: int):
        """Generate materials for a specific job."""
        if self.jobs_df is None:
            print("⚠️ Please run job search first!")
            return

        job_row = self.jobs_df.iloc[job_index]
        job_dict = {
            'Title': job_row['Title'],
            'Company': job_row['Company'],
            'URL': job_row['URL']
        }

        print("\n" + "=" * 60)
        print(f"📝 APPLICATION MATERIALS FOR: {job_dict['Title']}")
        print("=" * 60)

        # Cover letter
        print("\n--- COVER LETTER ---\n")
        cover_letter = self.helper.generate_cover_letter(self.cv_data, job_dict)
        print(cover_letter)

        # Resume bullets
        print("\n\n--- RESUME BULLETS ---\n")
        bullets = self.helper.generate_resume_bullets(self.cv_data, job_dict)
        for bullet in bullets:
            print(bullet)

        # Application email
        print("\n\n--- APPLICATION EMAIL ---\n")
        email = self.helper.generate_application_email(self.cv_data, job_dict)
        print(email)

        print("\n" + "=" * 60)

print("✅ Main App ready!")



✅ Main App ready!


In [11]:
# CELL 13: Run The APP
# SIMPLE UI VERSION - User-Friendly Interface

# If using Google Colab, enable download support
try:
    from google.colab import files
except:
    files = None

# ============================================================
# 🟣 UI COMPONENTS
# ============================================================

output = widgets.Output()

header_html = """
<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 30px; border-radius: 15px; text-align: center; margin-bottom: 20px;'>
    <h1 style='color: white; margin: 0; font-size: 2.5em;'>🚀 Remote Job Finder</h1>
    <p style='color: #f0f0f0; margin: 10px 0 0 0; font-size: 1.2em;'>
        Find your perfect remote job in minutes
    </p>
</div>
"""

instructions_html = """
<div style='background: #f8f9fa; padding: 20px; border-radius: 10px; margin-bottom: 20px; border-left: 4px solid #667eea;'>
    <h3 style='margin-top: 0; color: #667eea;'>📋 How to Use:</h3>
    <ol style='line-height: 2;'>
        <li><strong>Upload your CV</strong> (PDF, DOCX, or TXT format)</li>
        <li><strong>Enter job titles</strong> you're looking for</li>
        <li><strong>Set your preferences</strong> (remote only, etc.)</li>
        <li><strong>Click "Find Jobs"</strong> and wait for results</li>
        <li><strong>Download your matches</strong> and apply!</li>
    </ol>
</div>
"""

# STEP 1 — UPLOAD CV
upload_section = widgets.VBox([
    widgets.HTML("<h2 style='color: #667eea;'>📤 Step 1: Upload Your CV</h2>"),
    widgets.FileUpload(
        accept='.pdf,.docx,.doc,.txt',
        multiple=False,
        description='Choose CV',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
], layout=widgets.Layout(margin='20px 0'))

# STEP 2 — PREFERENCES
job_titles_input = widgets.Textarea(
    placeholder='e.g., data scientist, social media manager, ai engineer, content creator',
    description='Job Titles:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px', height='80px')
)

remote_only_checkbox = widgets.Checkbox(
    value=True,
    description='Remote jobs only'
)

min_match_slider = widgets.IntSlider(
    value=50, min=0, max=100, step=1,
    description='Match %:', continuous_update=False
)

posted_within_slider = widgets.IntSlider(
    value=14, min=1, max=30, step=1,
    description='Posted within (days):', continuous_update=False
)

preferences_section = widgets.VBox([
    widgets.HTML("<h2 style='color: #667eea;'>⚙️ Step 2: Set Your Preferences</h2>"),
    job_titles_input,
    widgets.HTML("<p style='color: #666; font-size: 0.9em;'>Separate multiple titles with commas</p>"),
    remote_only_checkbox,
    min_match_slider,
    posted_within_slider
], layout=widgets.Layout(margin='20px 0'))

# STEP 3 — BUTTONS
find_jobs_button = widgets.Button(
    description='🔍 Find Jobs', button_style='success',
    layout=widgets.Layout(width='200px', height='50px')
)
download_button = widgets.Button(
    description='📥 Download Results', button_style='primary',
    layout=widgets.Layout(width='200px', height='50px'), disabled=True
)
generate_materials_button = widgets.Button(
    description='📝 Generate Materials', button_style='info',
    layout=widgets.Layout(width='200px', height='50px'), disabled=True
)

buttons_section = widgets.HBox([
    find_jobs_button, download_button, generate_materials_button
], layout=widgets.Layout(margin='30px 0', justify_content='space-around'))

# PROGRESS + STATUS
progress = widgets.IntProgress(
    value=0, min=0, max=100, description='Progress:',
    bar_style='info', layout=widgets.Layout(width='600px', visibility='hidden')
)
status_label = widgets.HTML(value="<p style='color: #666;'>Ready...</p>")

# STATE
app = None
cv_uploaded = False

# ============================================================
# 🟣 EVENT HANDLERS
# ============================================================

def on_cv_upload(change):
    """Triggered when a CV is uploaded."""
    global app, cv_uploaded
    if not change['new']: return

    with output:
        clear_output()

    try:
        uploaded = list(change['new'].values())[0]
        filename = uploaded['metadata']['name']
        file_content = uploaded['content']

        status_label.value = "<p style='color:#ff9800;'>⏳ Parsing CV...</p>"
        progress.layout.visibility = 'visible'; progress.value = 20

        app = JobFinderApp(use_enhanced_parser=True)
        app.cv_data = app.parser.parse(filename, file_content)

        cv_uploaded = True
        progress.value = 100

        print("\n" + "="*70)
        print("✅ CV SUCCESSFULLY PARSED!")
        print("="*70)
        print(f"Email: {app.cv_data.get('email')}")
        print(f"Experience: {app.cv_data.get('years_experience')} years")
        print("Top Skills:")
        for s in app.cv_data.get('skills', [])[:10]: print(" -", s)

        status_label.value = "<p style='color:#4caf50;'>✅ CV Uploaded! Now find jobs.</p>"
        progress.layout.visibility = 'hidden'

    except Exception as e:
        status_label.value = f"<p style='color:#f44336;'>❌ Error: {e}</p>"
        progress.layout.visibility = 'hidden'


def on_find_jobs_click(b):
    """Triggered when Find Jobs is clicked."""
    global app
    if not cv_uploaded or app is None:
        status_label.value = "<p style='color:#f44336;'>❌ Upload CV first!</p>"
        return

    with output: clear_output()
    status_label.value = "<p style='color:#ff9800;'>⏳ Searching...</p>"
    progress.layout.visibility = 'visible'; progress.value = 20

    try:
        job_titles = [t.strip().lower() for t in job_titles_input.value.split(',') if t.strip()]
        if not job_titles:
            status_label.value = "<p style='color:#f44336;'>❌ Enter job titles.</p>"
            return

        # SEARCH
        jobs = app.scraper.scrape_all(keywords=job_titles)
        if not jobs:
            status_label.value = "<p style='color:#ff9800;'>⚠️ No jobs found. Try different keywords.</p>"
            progress.layout.visibility = 'hidden'
            return

        # NORMALIZE RESULTS
        jobs = [{
            'Title': j.get('title',''),
            'Company': j.get('company',''),
            'Location': j.get('location','Remote'),
            'Description': j.get('description',''),
            'Tags': j.get('tags', []),
            'URL': j.get('url',''),
            'Salary': j.get('salary','N/A'),
            'Posted': j.get('posted_date','N/A'),
            'Source': j.get('source','Unknown')
        } for j in jobs]

        progress.value = 50

        # MATCHING
        app.matcher = JobMatcher(app.cv_data)
        scored = app.matcher.score_jobs(jobs)
        app.jobs_df = pd.DataFrame(scored)

        progress.value = 70

        # FILTER
        job_filter = JobFilter({'job_titles': job_titles, 'remote_only': remote_only_checkbox.value})
        app.jobs_df = job_filter.apply_all_filters(
            app.jobs_df,
            max_days=posted_within_slider.value,
            min_score=min_match_slider.value,
        )

        progress.value = 100

        # RESULT
        if len(app.jobs_df) == 0:
            print("\n⚠️ No matching jobs. Loosen filters.")
            status_label.value = "<p style='color:#ff9800;'>⚠️ No results. Relax filters.</p>"
        else:
            print("\n🎉 FOUND", len(app.jobs_df), "MATCHES!")
            for i, r in app.jobs_df.head(10).iterrows():
                print(f"\n#{i+1} | {r['Title']} @ {r['Company']} — {r['Match Score']:.0f}%")

            status_label.value = f"<p style='color:#4caf50;'>✅ {len(app.jobs_df)} jobs found!</p>"
            download_button.disabled = False
            generate_materials_button.disabled = False

        progress.layout.visibility = 'hidden'

    except Exception as e:
        status_label.value = f"<p style='color:#f44336;'>❌ Error: {e}</p>"
        progress.layout.visibility = 'hidden'


def on_download_click(b):
    """Download results as Excel."""
    if app is None or app.jobs_df is None: return
    try:
        name = f"job_matches_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
        app.jobs_df.to_excel(name, index=False)

        if files: files.download(name)
        print("\n📥 Exported:", name)
        status_label.value = "<p style='color:#4caf50;'>📥 Downloaded!</p>"
    except Exception as e:
        print("❌ Error:", e)


# ============================================================
# 🟣 GENERATOR — (UNCHANGED EXCEPT INDENTATION)
# ============================================================

def on_generate_materials_click(b):
    """Generate cover letters, bullets & emails."""
    if app is None or app.jobs_df is None or len(app.jobs_df) == 0:
        return

    with output: clear_output()
    print("="*70, "\n📝 APPLICATION MATERIALS GENERATOR\n", "="*70)

    num_jobs_input = widgets.IntText(value=3, description='Number:', min=1)
    export_format = widgets.RadioButtons(
        options=['📄 DOCX', '📋 Display Here'],
        description='Output:'
    )
    go = widgets.Button(description='✨ Generate Now', button_style='success')

    def go_click(btn):
        with output: clear_output()
        num = min(num_jobs_input.value, len(app.jobs_df))
        print(f"Generating for top {num} jobs...\n")

        for i in range(num):
            row = app.jobs_df.iloc[i]
            job = {'Title': row['Title'], 'Company': row['Company'], 'URL': row['URL']}
            desc = row.get('Description', '')

            print("="*60)
            print(f"{i+1}. {job['Title']} at {job['Company']} ({row['Match Score']:.0f}%)")
            print("="*60)

            if "DOCX" in export_format.value:
                fname = app.helper.export_all_materials_docx(app.cv_data, job, desc)
                if files: files.download(fname)
                print("📄 DOCX:", fname)
            else:
                print("\n--- COVER LETTER ---")
                print(app.helper.generate_cover_letter(app.cv_data, job, desc))
                print("\n--- BULLETS ---")
                for b in app.helper.generate_resume_bullets(app.cv_data, job, desc):
                    print("•", b)

        status_label.value = "<p style='color:#4caf50;'>✨ Materials Ready!</p>"

    go.on_click(go_click)
    display(widgets.VBox([num_jobs_input, export_format, go]))

# ============================================================
# 🟣 CONNECT EVENTS + LAUNCH UI
# ============================================================

upload_section.children[1].observe(on_cv_upload, names='value')
find_jobs_button.on_click(on_find_jobs_click)
download_button.on_click(on_download_click)
generate_materials_button.on_click(on_generate_materials_click)

print("="*70)
print("🎨 LAUNCHING USER INTERFACE")
print("="*70)

display(HTML(header_html))
display(HTML(instructions_html))
display(upload_section)
display(preferences_section)
display(buttons_section)
display(progress)
display(status_label)
display(output)


🎨 LAUNCHING USER INTERFACE


IntProgress(value=0, bar_style='info', description='Progress:', layout=Layout(visibility='hidden', width='600p…

HTML(value="<p style='color: #666;'>Ready...</p>")

Output()


✅ CV SUCCESSFULLY PARSED!
Email: elizabethnnenna51@gmail.com
Experience: 1 years
Top Skills:
 - Insights & KPI Tracking

Platform Publishing & Optimization

Community Awareness & Social Listening

Customer
 - Social Media Strategy & Platform-Specific Planning

Content Calendar Management (Multi-Brand)

Content Adaptation & Repurposing

Caption Writing & Brand Voice Alignment

Analytics
 - ai
 - analytical
 - brand voice
 - canva
 - capcut
 - collaboration
 - communication
 - content calendar

🔍 Searching job platforms with JobSpy...

  📡 Fetching from JobSpy (LinkedIn, Indeed, Glassdoor, Google, Zip) for country: worldwide...
✅ Successfully fetched jobs for worldwide.
  📡 Fetching from RemoteOK (backup)...
    ✅ 25 from RemoteOK
  📡 Fetching from WeWorkRemotely (backup)...
    ✅ 25 from WWR

🎯 TOTAL UNIQUE JOBS: 45
   (Removed 5 duplicates)

🧠 Loading AI model...


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ AI model ready!

🎯 Scoring 45 jobs with enhanced matcher...
✅ Scoring complete!

🔍 APPLYING SMART CV-SPECIFIC FILTERS

🗓️  Filtering jobs posted within last 5 days...
   Kept 45/45 recent jobs

⭐ Filtering jobs with minimum 50% match...
   ✅ Kept 7/45 jobs (score threshold adjusted)

⚙️  Applying CV-based preference filters...
   ✅ Remote filter: 7/7 jobs
   ℹ️ Title filter skipped (insufficient jobs or no titles)

✅ FILTERING COMPLETE: 7/45 jobs match your criteria
   🔝 Best match: 98.3%
   📊 Average: 77.2%
   📅 Date range: 2-3 days ago


🎉 FOUND 7 MATCHES!

#1 | Social Media Manager Platform @ Digital Media Management — 98%

#2 | Client Success Manager Coupa @ CrossCountry Consulting — 96%

#3 | Virtual Operations Manager @ Women Builders Council — 78%

#4 | Senior Business Analyst @ Xpansiv — 75%

#5 | Product Designer @ Ahrefs — 71%

#6 | Project Coordinator Contract @ INFUSE — 63%

#7 | Client Success Manager @ Offshore Launch — 59%
